In [10]:
import sys

!{sys.executable} -m pip uninstall -y numpy pandas scipy
!{sys.executable} -m pip install --upgrade pip setuptools wheel
!{sys.executable} -m pip install --no-cache-dir "numpy<2" pandas scipy

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Found existing installation: pandas 2.3.3
Uninstalling pandas-2.3.3:
  Successfully uninstalled pandas-2.3.3
Found existing installation: scipy 1.15.3
Uninstalling scipy-1.15.3:
  Successfully uninstalled scipy-1.15.3
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.6/20.6 MB 8.5 MB/s  0:00:02 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 19.3 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.7/38.7 MB 22.0 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [pandas]2m2/3 [pandas]


In [11]:
import sys
print(sys.executable)

/Users/tiago/Downloads/alzheimers-detection-methodologies-organized/.venv/bin/python


In [2]:
import sys
!{sys.executable} -m pip install matplotlib scikit-learn tensorflow

  Using cached matplotlib-3.10.8-cp310-cp310-macosx_10_12_x86_64.whl.metadata (52 kB)
  Using cached scikit_learn-1.7.2-cp310-cp310-macosx_10_9_x86_64.whl.metadata (11 kB)
  Using cached contourpy-1.3.2-cp310-cp310-macosx_10_9_x86_64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.62.1-cp310-cp310-macosx_10_9_x86_64.whl.metadata (117 kB)
  Using cached kiwisolver-1.5.0-cp310-cp310-macosx_10_9_x86_64.whl.metadata (5.1 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached matplotlib-3.10.8-cp310-cp310-macosx_10_12_x86_64.whl (8.2 MB)
Using cached scikit_learn-1.7.2-cp310-cp310-macosx_10_9_x86_64.whl (9.3 MB)
Using cached contourpy-1.3.2-cp310-cp310-macosx_10_9_x86_64.whl (268 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
Using cached fonttools-4.62.1-cp31

In [1]:
import numpy as np
import pandas as pd
import glob
import re
import shutil
import random
import tensorflow as tf
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA 
from sklearn.decomposition import KernelPCA

2026-04-01 16:51:56.787761: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
print(np.__version__)
print(pd.__version__)
print(tf.__version__)

1.26.4
2.3.3
2.16.2


In [6]:
import os
print(os.getcwd())

/Users/tiago/Downloads/alzheimers-detection-methodologies-organized/notebooks/ml_based_models


In [9]:
!ls
!ls ..
!ls ../..

01_AlzheimerDetection_Data_Preprocessing.ipynb
02_AlzheimersDetection_ML_Models.ipynb
Environment            few_shot_similarity_nn ml_based_models
Alzheimer_Environment_Python_3.12.venv model
README.md                              notebooks
data                                   results


In [10]:
!unzip ../../data/raw/dataset_non_augmented.zip -d ../../data/raw

Archive:  ../../data/raw/dataset_non_augmented.zip
   creating: /Users/tiago/Downloads/alzheimers-detection-methodologies-organized/data/raw/DataBase/SETA
  inflating: ../../data/raw/DataBase/SETA/healthy_open1.csv  
  inflating: ../../data/raw/DataBase/SETA/healthy_open10.csv  
  inflating: ../../data/raw/DataBase/SETA/healthy_open11.csv  
  inflating: ../../data/raw/DataBase/SETA/healthy_open12.csv  
  inflating: ../../data/raw/DataBase/SETA/healthy_open2.csv  
  inflating: ../../data/raw/DataBase/SETA/healthy_open3.csv  
  inflating: ../../data/raw/DataBase/SETA/healthy_open4.csv  
  inflating: ../../data/raw/DataBase/SETA/healthy_open5.csv  
  inflating: ../../data/raw/DataBase/SETA/healthy_open6.csv  
  inflating: ../../data/raw/DataBase/SETA/healthy_open7.csv  
  inflating: ../../data/raw/DataBase/SETA/healthy_open8.csv  
  inflating: ../../data/raw/DataBase/SETA/healthy_open9.csv  
   creating: /Users/tiago/Downloads/alzheimers-detection-methodologies-organized/data/raw/DataBase

In [13]:
import glob

# Usando o caminho relativo que aponta para a pasta onde os dados foram extraídos
files = glob.glob("../../data/raw/DataBase/*/*")

In [14]:
len(files)

48

Cleaning up the files which has non-int or non-float datatype which will be replaced by the previous timestep or its next time step

In [15]:
def clean(path):
    df = pd.read_csv(path)
    for column in df.columns:
        if df[column].dtype == 'object':
            print("Sample : ",path," feature : ",column," is uncleaned")
            df[column] = pd.to_numeric(df[column], errors='coerce')
            df[column] = df[column].fillna(method='ffill')
            df[column] = df[column].fillna(method='bfill')
    df = df.iloc[:1024,:]
    df.to_csv(path, index=False)

In [16]:
for i in files:
    clean(i)

Sample :  ../../data/raw/DataBase/SETB/healthy_closed6.csv  feature :  14  is uncleaned
Sample :  ../../data/raw/DataBase/SETB/healthy_closed2.csv  feature :  16  is uncleaned
Sample :  ../../data/raw/DataBase/SETC/alzeimer_open10.csv  feature :  0  is uncleaned


/var/folders/yp/kd81cw5n6y9cdv4bxmrqgpf80000gn/T/ipykernel_82209/357744548.py:7: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df[column] = df[column].fillna(method='ffill')
/var/folders/yp/kd81cw5n6y9cdv4bxmrqgpf80000gn/T/ipykernel_82209/357744548.py:8: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df[column] = df[column].fillna(method='bfill')
/var/folders/yp/kd81cw5n6y9cdv4bxmrqgpf80000gn/T/ipykernel_82209/357744548.py:7: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df[column] = df[column].fillna(method='ffill')
/var/folders/yp/kd81cw5n6y9cdv4bxmrqgpf80000gn/T/ipykernel_82209/357744548.py:8: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instea

Sample :  ../../data/raw/DataBase/SETC/alzeimer_open8.csv  feature :  0  is uncleaned


/var/folders/yp/kd81cw5n6y9cdv4bxmrqgpf80000gn/T/ipykernel_82209/357744548.py:7: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df[column] = df[column].fillna(method='ffill')
/var/folders/yp/kd81cw5n6y9cdv4bxmrqgpf80000gn/T/ipykernel_82209/357744548.py:8: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df[column] = df[column].fillna(method='bfill')


Sample :  ../../data/raw/DataBase/SETC/alzeimer_open6.csv  feature :  0  is uncleaned
Sample :  ../../data/raw/DataBase/SETC/alzeimer_open5.csv  feature :  0  is uncleaned


/var/folders/yp/kd81cw5n6y9cdv4bxmrqgpf80000gn/T/ipykernel_82209/357744548.py:7: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df[column] = df[column].fillna(method='ffill')
/var/folders/yp/kd81cw5n6y9cdv4bxmrqgpf80000gn/T/ipykernel_82209/357744548.py:8: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df[column] = df[column].fillna(method='bfill')
/var/folders/yp/kd81cw5n6y9cdv4bxmrqgpf80000gn/T/ipykernel_82209/357744548.py:7: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df[column] = df[column].fillna(method='ffill')
/var/folders/yp/kd81cw5n6y9cdv4bxmrqgpf80000gn/T/ipykernel_82209/357744548.py:8: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instea

Sample :  ../../data/raw/DataBase/SETD/alzeimer_closed12.csv  feature :  18  is uncleaned


/var/folders/yp/kd81cw5n6y9cdv4bxmrqgpf80000gn/T/ipykernel_82209/357744548.py:7: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df[column] = df[column].fillna(method='ffill')
/var/folders/yp/kd81cw5n6y9cdv4bxmrqgpf80000gn/T/ipykernel_82209/357744548.py:8: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df[column] = df[column].fillna(method='bfill')


Sample :  ../../data/raw/DataBase/SETA/healthy_open2.csv  feature :  16  is uncleaned
Sample :  ../../data/raw/DataBase/SETA/healthy_open11.csv  feature :  14  is uncleaned


/var/folders/yp/kd81cw5n6y9cdv4bxmrqgpf80000gn/T/ipykernel_82209/357744548.py:7: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df[column] = df[column].fillna(method='ffill')
/var/folders/yp/kd81cw5n6y9cdv4bxmrqgpf80000gn/T/ipykernel_82209/357744548.py:8: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df[column] = df[column].fillna(method='bfill')
/var/folders/yp/kd81cw5n6y9cdv4bxmrqgpf80000gn/T/ipykernel_82209/357744548.py:7: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df[column] = df[column].fillna(method='ffill')
/var/folders/yp/kd81cw5n6y9cdv4bxmrqgpf80000gn/T/ipykernel_82209/357744548.py:8: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instea

In [17]:
files = glob.glob("/content/DataBase/*/*")
len(files)

0

In [19]:
import shutil

# Apontando para onde os dados foram extraídos
caminho_origem = "../../data/raw/DataBase"

# Apontando para onde a cópia de segurança deve ser criada
caminho_destino = "../../data/raw/DataBaseOriginal"

# Executando a cópia
shutil.copytree(caminho_origem, caminho_destino)

print("Cópia de segurança criada com sucesso!")

Cópia de segurança criada com sucesso!


# Data Augmentation Techniques : 
  1.) **Shifting and Noising :** Shifting in time series data refers to a
transformation of the data where the entire series is moved forward or backward in time by a certain number of periods. This can be useful in analyzing time series data, as it can help to identify patterns or trends that may not be immediately apparent when looking at the original data.Noising adds random noise to the shifted data so that the learning model will learn to ignore noisy information and filter out the relevant info from the data.

 2.) **Rolling Mean :** Rolling mean is a technique used to smooth out the data by taking the average of a fixed window of data points. This technique can be useful for reducing noise in the data and identifying trends that are relevant.


1.) Shifting 

In [21]:
def positive_shift(path,replace = False,value = 0):
    df = pd.read_csv(path)
    shift = random.randint(3,15)
    df = df.shift(shift)
    if replace:
       df = df.fillna(df.mean())
    df = df.fillna(0)      
    deviation = df.std().tolist()
    noise = np.random.normal([0]*19,deviation,19)
    df += noise/10
    return df

def negative_shift(path,replace = False,value = 0):
    df = pd.read_csv(path)
    shift = random.randint(-15,-3)
    df = df.shift(shift)
    if replace:
       df = df.fillna(df.mean())
    df = df.fillna(0)   
    deviation = df.std().tolist()
    noise = np.random.normal([0]*19,deviation,19)
    df += noise/10
    return df    

def shift(path,label,file_no,target_dir):
    pn = random.randint(0,1)
    if pn == 1:
        file_name = label+str(file_no)+".csv"
        file_name = target_dir+"/"+file_name
        df = positive_shift(path)
        print(path," -> ",file_name," Positive Shift")
        df.to_csv(file_name,index = False)
    else:
        file_name = label+str(file_no)+".csv"
        file_name = target_dir+"/"+file_name
        df = negative_shift(path)
        print(path," -> ",file_name," Negative Shift")
        df.to_csv(file_name,index = False)    


def add_shift_per_sample(path,label):
    file_no = 25
    all_files = glob.glob(path+"/*")
    for i in all_files:
        shift(i,label,file_no,path)
        file_no += 1
   

In [22]:
add_shift_per_sample("/content/DataBase/SETA",'healthy_open')

In [23]:
add_shift_per_sample("/content/DataBase/SETB",'healthy_closed')

In [24]:
add_shift_per_sample("/content/DataBase/SETC",'alzeimer_open')

In [25]:
add_shift_per_sample("/content/DataBase/SETD",'alzeimer_closed')

2.) Rolling Mean

In [26]:
def mean_roll(path,label,file_no,target_dir):
    df = pd.read_csv(path)
    window = random.randint(3,8)
    df = df.rolling(window = 5,center = True).mean().fillna(0)
    file_name = label+str(file_no)+".csv"
    file_name = target_dir+"/"+file_name
    print(path," -> ",file_name)
    df = df.fillna(method = 'ffill')
    df = df.fillna(0)
    df.to_csv(file_name,index = False)

def add_rolling_per_sample(path,label):
    file_no = 49
    all_files = glob.glob(path+"/*")
    for i in all_files:
        mean_roll(i,label,file_no,path)
        file_no += 1

In [27]:
add_rolling_per_sample("/content/DataBase/SETA",'healthy_open')

In [28]:
add_rolling_per_sample("/content/DataBase/SETB",'healthy_closed')

In [29]:
add_rolling_per_sample("/content/DataBase/SETC",'alzeimer_open')

In [30]:
add_rolling_per_sample("/content/DataBase/SETD",'alzeimer_closed')

In [31]:
shutil.make_archive('Augmented', 'zip','/content/DataBase')

'/Users/tiago/Downloads/alzheimers-detection-methodologies-organized/notebooks/ml_based_models/Augmented.zip'

**Splitting data to train and test**

In [34]:
import glob

# 1. Primeiro, garantimos que a variável 'files' tem os dados locais corretos
files = glob.glob("../../data/raw/DataBase/*/*")

def return_splits():
    # 2. Corrigimos o caminho do Colab aqui também
    # Obs: Se você tiver uma pasta separada para dados aumentados depois, 
    # você precisará alterar este caminho para apontar para ela.
    augmented_files = glob.glob("../../data/raw/DataBase/*/*") 
    
    train_files = list(set(augmented_files) - set(files))
    test_files = files.copy() # .copy() para evitar alterar a lista original por acidente
    
    for i in range(0, len(test_files), 12):
        train_files += test_files[i:i+4]
        
    test_files = list(set(test_files) - set(train_files))
    
    return train_files, test_files

# 3. Executamos a função
train, test = return_splits()

In [35]:
len(train),len(test)

(16, 32)

# Compressing Time Series Data 
PCA can be used to compress time series data of many time steps to a vector of length musch lesser by identifying the most important patterns or features in the data and representing it using a smaller set of variables or dimensions, called principal components.
Suppose we have a time series dataset with 10,000 time steps and each time step has multiple variables. We can apply PCA to this data to identify the most important patterns or features across all the variables. PCA will generate a set of principal components, with each component representing a linear combination of the original variables that captures the most variance in the data.


In [39]:
def decompose(wave_data):    
    time_series_data = wave_data.T
    kpca = KernelPCA(n_components=5, kernel='rbf', gamma=0.1)
    compressed_data = kpca.fit_transform(time_series_data).mean(axis = 1)
    return compressed_data.tolist()

In [36]:
def handle_nan(df_f):
    df = pd.read_csv(df_f)
    count = 0
    if df.isnull().sum().sum() != 0:
        print(df_f,end = " ")
        while df.isnull().sum().sum() != 0:
            count = count + 1
            df = df.fillna(method = 'ffill')
            df = df.fillna(method = 'bfill')
        print("   " , count, "null values found and taken care of","\n")    
    return df

In [37]:
def create_ds(all_files):
    dataset = []
    for file in all_files:
        df = handle_nan(file)
        scaler = StandardScaler()
        df = scaler.fit_transform(df)
        sample = decompose(df)

        if "closed" in file:
            sample.append("closed")
        elif "open" in file:
            sample.append("open")

        if "healthy" in file:
            sample.append("healthy")
        elif "alzeimer" in file:
            sample.append("alzeimer")

        dataset.append(sample)
    return pd.DataFrame(dataset)        

In [41]:
train_df = create_ds(train)

/content/DataBase/SETC/alzeimer_open8.csv     1 null values found and taken care of 



In [42]:
train_df.to_csv("train.csv",index= False)

In [43]:
test_df = create_ds(test)

/content/DataBase/SETC/alzeimer_open5.csv     1 null values found and taken care of 

/content/DataBase/SETC/alzeimer_open6.csv     1 null values found and taken care of 

/content/DataBase/SETC/alzeimer_open10.csv     1 null values found and taken care of 



In [44]:
test_df.to_csv("test.csv",index= False)